# Granite Rationale Training (from JSONL)

Simple notebook to train Granite 3.2-2B on pre-generated rationale JSONL.

**Setup on Kaggle:**
1. Add dataset `gigibot/rationale-semeval2026` as Input
2. Enable GPU T4 x2
3. Run all cells

In [ ]:
!pip -q install -U transformers datasets accelerate bitsandbytes peft pandas scikit-learn

In [ ]:
### Cell 1: Setup & Config
import os
import json
import random
from pathlib import Path
from collections import Counter

import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType

# Environment detection
IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB = 'COLAB_GPU' in os.environ
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment: Kaggle={IN_KAGGLE}, Colab={IN_COLAB}, Device={DEVICE}')

# ============ CONFIG ============
# Input JSONL - change this path!
TRAINING_JSONL = Path('/kaggle/input/rationale-semeval2026/RationaleTraining_raw.jsonl')

# Model
BASE_MODEL = 'ibm-granite/granite-3.2-2b-instruct'
LOAD_8BIT = True

# Training
EPOCHS = 2
BATCH_SIZE = 1
GRADIENT_ACCUM = 8
LEARNING_RATE = 2e-4
MAX_LENGTH = 1024
VALIDATION_SPLIT = 0.1
BALANCE_MODE = 'upsample'  # 'upsample', 'downsample', or None

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Output
OUTPUT_DIR = Path('/kaggle/working/granite_lora_trained')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Training JSONL: {TRAINING_JSONL}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
### Cell 2: Load and Parse JSONL
import re

LABEL_MAP = {
    'Clear Reply': 'Direct Reply',
    'Clear Non-Reply': 'Direct Non-Reply',
    'Ambivalent': 'Indirect',
    'Ambivalent Reply': 'Indirect',
}

def map_label(raw):
    v = str(raw).strip()
    return LABEL_MAP.get(v, v if v in ['Direct Reply', 'Direct Non-Reply', 'Indirect'] else 'Indirect')

def parse_jsonl_record(rec):
    """Parse a JSONL record into Q, A, reasoning, label."""
    inp = rec.get('input', '')
    out = rec.get('output', '')
    
    # Parse Q/A from input
    m = re.match(r'Q:\s*(.*?)\nA:\s*(.*)', inp, re.DOTALL)
    if m:
        q, a = m.group(1).strip(), m.group(2).strip()
    else:
        q, a = '', inp
    
    # Parse reasoning and verdict from output
    reasoning = ''
    verdict = ''
    think = re.search(r'<think>(.*?)</think>', out, re.DOTALL | re.IGNORECASE)
    if think:
        reasoning = think.group(1).strip()
    vm = re.search(r'Verdict:\s*([^\n]+)', out, re.IGNORECASE)
    if vm:
        verdict = vm.group(1).strip()
    
    return {
        'question': q,
        'answer': a,
        'reasoning': reasoning,
        'label': map_label(verdict),
    }

# Load JSONL
if not TRAINING_JSONL.exists():
    raise FileNotFoundError(f'JSONL not found: {TRAINING_JSONL}. Add dataset as Kaggle Input.')

rows = []
with open(TRAINING_JSONL, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        parsed = parse_jsonl_record(rec)
        if parsed['question'] and parsed['answer'] and parsed['label']:
            rows.append(parsed)

print(f'Loaded {len(rows)} examples')
label_counts = Counter(r['label'] for r in rows)
print(f'Label distribution: {dict(label_counts)}')

In [ ]:
### Cell 3: Balance and Split Data

def upsample_to_max(rows):
    """Upsample minority classes to match majority."""
    by_label = {}
    for r in rows:
        by_label.setdefault(r['label'], []).append(r)
    max_count = max(len(v) for v in by_label.values())
    balanced = []
    for label, items in by_label.items():
        if len(items) < max_count:
            upsampled = items * (max_count // len(items)) + random.sample(items, max_count % len(items))
            balanced.extend(upsampled)
        else:
            balanced.extend(items)
    random.shuffle(balanced)
    return balanced

def downsample_to_min(rows):
    """Downsample majority classes to match minority."""
    by_label = {}
    for r in rows:
        by_label.setdefault(r['label'], []).append(r)
    min_count = min(len(v) for v in by_label.values())
    balanced = []
    for label, items in by_label.items():
        balanced.extend(random.sample(items, min_count))
    random.shuffle(balanced)
    return balanced

# Balance
if BALANCE_MODE == 'upsample':
    rows = upsample_to_max(rows)
    print(f'After upsampling: {len(rows)} examples')
elif BALANCE_MODE == 'downsample':
    rows = downsample_to_min(rows)
    print(f'After downsampling: {len(rows)} examples')

# Split
random.shuffle(rows)
n_val = int(len(rows) * VALIDATION_SPLIT)
val_rows = rows[:n_val]
train_rows = rows[n_val:]
print(f'Train: {len(train_rows)}, Validation: {len(val_rows)}')

In [ ]:
### Cell 4: Load Model and Tokenizer

print(f'Loading model: {BASE_MODEL}')

# Quantization config
quant_config = None
if LOAD_8BIT and torch.cuda.is_available():
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

# LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if hasattr(model, 'gradient_checkpointing_enable'):
    model.gradient_checkpointing_enable()

print('✅ Model loaded with LoRA')

In [ ]:
### Cell 5: Prepare Dataset

def build_prompt(q, a):
    return f"""You are analyzing political interview answers for clarity classification.

Question: {q}
Answer: {a}

Analyze the answer step-by-step:
1. Does it directly address the question?
2. Is it evasive or indirect?
3. Does it decline to answer?

Provide your reasoning and then classify as one of:
- "Direct Reply": Directly answers the question
- "Direct Non-Reply": Explicitly declines or claims inability to answer  
- "Indirect": Evasive, indirect, or partially answers

Respond in JSON format:
{{
  "reasoning": "Your step-by-step analysis...",
  "label": "Direct Reply|Direct Non-Reply|Indirect"
}}"""

def build_assistant(reasoning, label):
    return json.dumps({'reasoning': reasoning, 'label': label}, ensure_ascii=False)

def tokenize_example(row):
    user_msg = build_prompt(row['question'], row['answer'])
    assistant_msg = build_assistant(row['reasoning'], row['label'])
    
    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': assistant_msg},
    ]
    
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except:
        text = f"User: {user_msg}\n\nAssistant: {assistant_msg}"
    
    # Tokenize
    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
        return_tensors=None,
    )
    
    # Labels = input_ids (for causal LM)
    encoded['labels'] = encoded['input_ids'].copy()
    
    return encoded

# Create datasets
train_data = [tokenize_example(r) for r in train_rows]
val_data = [tokenize_example(r) for r in val_rows]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f'Train dataset: {len(train_dataset)} examples')
print(f'Val dataset: {len(val_dataset)} examples')

In [ ]:
### Cell 6: Train!

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to='none',
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print('🚀 Starting training...')
trainer.train()
print('✅ Training complete!')

# Save final model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'💾 Model saved to: {OUTPUT_DIR}')

In [ ]:
### Cell 7: Quick Evaluation
import re

model.eval()

def predict(q, a):
    prompt = build_prompt(q, a)
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Parse label from response
    try:
        data = json.loads(response)
        return data.get('label', 'Indirect')
    except:
        for lbl in ['Direct Reply', 'Direct Non-Reply', 'Indirect']:
            if lbl.lower() in response.lower():
                return lbl
        return 'Indirect'

# Evaluate on validation set
n_eval = min(20, len(val_rows))
correct = 0
print(f'Evaluating on {n_eval} examples...')

for i, row in enumerate(val_rows[:n_eval]):
    pred = predict(row['question'], row['answer'])
    true = row['label']
    match = pred == true
    correct += match
    print(f'{i+1}. True={true}, Pred={pred}, {"✅" if match else "❌"}')

print(f'\nAccuracy: {correct}/{n_eval} = {correct/n_eval:.1%}')

In [ ]:
### Cell 8: Push to HuggingFace (optional)
from huggingface_hub import HfApi, login

HF_REPO = 'gigibot/granite-clarity-lora'  # Change this!
hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    api = HfApi()
    print(f'📤 Pushing to {HF_REPO}...')
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=HF_REPO,
        repo_type='model',
    )
    print(f'✅ Pushed to https://huggingface.co/{HF_REPO}')
else:
    print('⚠️ No HF_TOKEN found. Add to Kaggle Secrets to push model.')
    print(f'Model saved locally at: {OUTPUT_DIR}')